# Phase 2: Chest X-Ray Computer Vision Classifier

This notebook is designed to be uploaded to **Google Colab** to utilize a free cloud GPU for training a PyTorch Convolutional Neural Network (CNN) on medical imaging data.

**Objective:** Classify Chest X-Rays as Normal vs. Pneumonia.

In [ ]:
# Install Kaggle API to download the dataset directly into the cloud environment
!pip install -q kaggle

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

print(f"PyTorch version: {torch.__version__}")

# Check if Cloud GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Data Augmentation & Preprocessing
We use `torchvision.transforms` to resize the X-rays, convert them to PyTorch tensors, and apply basic data augmentation (like horizontal flips) to prevent overfitting.

In [ ]:
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Note: In Colab, you would load the downloaded dataset here
# train_dataset = datasets.ImageFolder(root='path/to/train', transform=transform_train)
# train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

## 2. Model Architecture
We will load a pre-trained ResNet18 model (Transfer Learning) and modify the final fully connected layer for binary classification (Normal vs Pneumonia).

In [ ]:
# Load pre-trained ResNet18
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Freeze early layers to retain generic image features
for param in model.parameters():
    param.requires_grad = False

# Replace the final layer for 2 classes
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)

model = model.to(device)
print(model)

## 3. Training Loop

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

epochs = 5

# Placeholder training loop for demonstration
def train_model(model, criterion, optimizer, epochs=5):
    print("Starting training on Cloud GPU...")
    # for epoch in range(epochs):
    #     model.train()
    #     for inputs, labels in train_loader:
    #         inputs, labels = inputs.to(device), labels.to(device)
    #         optimizer.zero_grad()
    #         outputs = model(inputs)
    #         loss = criterion(outputs, labels)
    #         loss.backward()
    #         optimizer.step()
    print("Training complete! Model saved.")

train_model(model, criterion, optimizer)

# Save the weights to be exported to the Node.js backend
torch.save(model.state_dict(), 'pneumonia_cnn_weights.pth')